**Nombre:** Benjamin Naranjo
**RUT:**  21687575-3
**Fecha:** 17-06-2026

In [1]:
import sqlite3
import os
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = "clima_lab.db"
CSV_PATH = "S15_registros_climaticos.csv"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS registros (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ciudad TEXT NOT NULL,
            temp_c REAL,
            humedad INTEGER,
            precip_mm REAL,
            viento_kmh REAL,
            fecha TEXT
        )
    ''')
    conn.commit()
    print("Tabla 'registros' creada.\n")

    cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name='registros'")
    nombre_tabla, sql_generado = cursor.fetchone()
    print(f"Tabla encontrada en sqlite_master: '{nombre_tabla}'")
    print("SQL generado:")
    print(sql_generado)
    print()

    cursor.execute("PRAGMA table_info(registros)")
    columnas = cursor.fetchall()
    print("Columnas:")
    for col in columnas:
        print(f"  {col[1]:<12} {col[2]}")


Tabla 'registros' creada.

Tabla encontrada en sqlite_master: 'registros'
SQL generado:
CREATE TABLE registros (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ciudad TEXT NOT NULL,
            temp_c REAL,
            humedad INTEGER,
            precip_mm REAL,
            viento_kmh REAL,
            fecha TEXT
        )

Columnas:
  id           INTEGER
  ciudad       TEXT
  temp_c       REAL
  humedad      INTEGER
  precip_mm    REAL
  viento_kmh   REAL
  fecha        TEXT


In [3]:
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    registro_santiago = ('Santiago', 18.5, 72, 1.2, 15.3, '2026-06-15')
    cursor.execute(
        "INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        registro_santiago
    )
    print(f"Registro de Santiago insertado con id = {cursor.lastrowid}")

    nuevos_registros = [
        ('Valparaíso', 15.2, 80, 3.5, 22.1, '2026-06-15'),
        ('Concepción', 12.8, 85, 8.0, 18.7, '2026-06-15'),
        ('Temuco',      9.3, 90, 12.5, 25.0, '2026-06-15'),
        ('La Serena',  17.8, 55,  0.0, 12.4, '2026-06-15'),
    ]
    cursor.executemany(
        "INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        nuevos_registros
    )
    print(f"{cursor.rowcount} registros adicionales insertados con executemany().")
    conn.commit()
    cursor.execute("SELECT COUNT(*) FROM registros")
    total = cursor.fetchone()[0]
    print(f"Total de registros en la tabla: {total}")


Registro de Santiago insertado con id = 1
4 registros adicionales insertados con executemany().
Total de registros en la tabla: 5



## Ejercicio 3: Consultas SELECT con Filtros y Agregaciones

In [4]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    print("Consulta 1 — Listado completo")
    print("")
    cursor.execute("SELECT ciudad, temp_c, fecha FROM registros")
    filas = cursor.fetchall()
    print(f"{'Ciudad':<15}{'Temp (°C)':<12}{'Fecha':<12}")
    for fila in filas:
        print(f"{fila['ciudad']:<15}{fila['temp_c']:<12}{fila['fecha']:<12}")
    print()
    print("Consulta 2 — Ciudad más cálida")
    cursor.execute("SELECT ciudad, temp_c FROM registros ORDER BY temp_c DESC")
    mas_calida = cursor.fetchone()
    print(f"La ciudad más cálida es {mas_calida['ciudad']} con {mas_calida['temp_c']}°C")
    print()
    print("Consulta 3 — Ciudades con temperatura menor a 13°C")
    umbral = 13.0
    cursor.execute("SELECT ciudad, temp_c FROM registros WHERE temp_c < ?", (umbral,))
    frias = cursor.fetchall()
    for fila in frias:
        print(f"  {fila['ciudad']:<15} {fila['temp_c']}°C")
    print()
    print("Consulta 4 — Estadísticas por ciudad")
    cursor.execute('''
        SELECT ciudad,
               AVG(temp_c)  AS temp_promedio,
               MIN(humedad) AS humedad_min,
               MAX(humedad) AS humedad_max
        FROM registros
        GROUP BY ciudad
        ORDER BY temp_promedio DESC
    ''')
    stats = cursor.fetchall()
    print(f"{'Ciudad':<15}{'Temp prom.':<12}{'Hum. min':<10}{'Hum. max':<10}")
    for fila in stats:
        print(f"{fila['ciudad']:<15}{fila['temp_promedio']:<12.1f}{fila['humedad_min']:<10}{fila['humedad_max']:<10}")


Consulta 1 — Listado completo

Ciudad         Temp (°C)   Fecha       
Santiago       18.5        2026-06-15  
Valparaíso     15.2        2026-06-15  
Concepción     12.8        2026-06-15  
Temuco         9.3         2026-06-15  
La Serena      17.8        2026-06-15  

Consulta 2 — Ciudad más cálida
La ciudad más cálida es Santiago con 18.5°C

Consulta 3 — Ciudades con temperatura menor a 13°C
  Concepción      12.8°C
  Temuco          9.3°C

Consulta 4 — Estadísticas por ciudad
Ciudad         Temp prom.  Hum. min  Hum. max  
Santiago       18.5        72        72        
La Serena      17.8        55        55        
Valparaíso     15.2        80        80        
Concepción     12.8        85        85        
Temuco         9.3         90        90        


## Ejercicio 4: UPDATE y DELETE con Validación

In [5]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    try:
        print("Operación 1 — Actualizar temperatura de Valparaíso")
        cursor.execute(
            "UPDATE registros SET temp_c = ? WHERE ciudad = ?",
            (16.0, 'Valparaíso')
        )
        conn.commit()
        print(f"  Filas modificadas: {cursor.rowcount}")

        cursor.execute("SELECT ciudad, temp_c FROM registros WHERE ciudad = ?", ('Valparaíso',))
        print(f"  Verificación: {dict(cursor.fetchone())}")
        print()
        print("Operación 2 — Sumar 5 puntos de humedad a ciudades con temp_c < 10°C")
        cursor.execute(
            "UPDATE registros SET humedad = humedad + 5 WHERE temp_c < ?",
            (10.0,)
        )
        conn.commit()
        print(f"  Filas afectadas: {cursor.rowcount}")
        print()
        print("Operación 3 — Eliminar registros de Temuco")
        cursor.execute("DELETE FROM registros WHERE ciudad = ?", ('Temuco',))
        conn.commit()
        print(f"  Filas eliminadas: {cursor.rowcount}")

        cursor.execute("SELECT COUNT(*) FROM registros WHERE ciudad = ?", ('Temuco',))
        restantes = cursor.fetchone()[0]
        print(f"  Verificación: registros de Temuco restantes = {restantes}")

        cursor.execute("SELECT COUNT(*) FROM registros")
        total_final = cursor.fetchone()[0]
        print(f"  Total de registros en la tabla tras las operaciones: {total_final}")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"Error al ejecutar la operación: {e}")


Operación 1 — Actualizar temperatura de Valparaíso
  Filas modificadas: 1
  Verificación: {'ciudad': 'Valparaíso', 'temp_c': 16.0}

Operación 2 — Sumar 5 puntos de humedad a ciudades con temp_c < 10°C
  Filas afectadas: 1

Operación 3 — Eliminar registros de Temuco
  Filas eliminadas: 1
  Verificación: registros de Temuco restantes = 0
  Total de registros en la tabla tras las operaciones: 4


## Ejercicio 5: Pipeline Completo CSV → Limpieza → SQLite → Análisis SQL

In [6]:
df_raw = pd.read_csv(CSV_PATH)

print("Información general del dataset:")
df_raw.info()

print()
print("Total de valores nulos por columna:")
print(df_raw.isna().sum())

filas_originales = len(df_raw)
print(f"\nTotal de filas cargadas originalmente: {filas_originales}")


Información general del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ciudad      208 non-null    object 
 1   temp_c      194 non-null    float64
 2   humedad     198 non-null    float64
 3   precip_mm   208 non-null    float64
 4   viento_kmh  208 non-null    float64
 5   fecha       208 non-null    object 
dtypes: float64(4), object(2)
memory usage: 9.9+ KB

Total de valores nulos por columna:
ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

Total de filas cargadas originalmente: 208


In [7]:
df_limpio = df_raw.copy()

# 1) Normalizar nombres de ciudad
df_limpio['ciudad'] = df_limpio['ciudad'].str.title()

# 2) Estandarizar fechas a formato YYYY-MM-DD
df_limpio['fecha'] = pd.to_datetime(
    df_limpio['fecha'], errors='coerce', format='mixed', dayfirst=False
).dt.strftime('%Y-%m-%d')

filas_antes_dedup = len(df_limpio)

# 3) Eliminar duplicados por ciudad + fecha 
df_limpio = df_limpio.drop_duplicates(subset=['ciudad', 'fecha'])
duplicados_eliminados = filas_antes_dedup - len(df_limpio)

filas_antes_outliers = len(df_limpio)

# 4) Filtrar valores atípicos de temperatura fuera de [-5, 50]
df_limpio = df_limpio[df_limpio['temp_c'].between(-5, 50) | df_limpio['temp_c'].isna()]
outliers_eliminados = filas_antes_outliers - len(df_limpio)

print(f"Filas originales:            {filas_originales}")
print(f"Duplicados eliminados:       {duplicados_eliminados}")
print(f"Valores atípicos eliminados: {outliers_eliminados}")
print(f"Filas tras la limpieza:      {len(df_limpio)}")

df_limpio.head(10)


Filas originales:            208
Duplicados eliminados:       13
Valores atípicos eliminados: 5
Filas tras la limpieza:      190


,ciudad,temp_c,humedad,precip_mm,viento_kmh,fecha
0,Rancagua,20.3,65.0,2.7,16.0,2026-05-14
1,La Serena,NaN,62.0,5.2,19.4,2026-01-13
2,La Serena,23.2,87.0,2.3,10.1,2026-03-02
3,Iquique,15.8,84.0,0.7,25.9,2026-03-21
4,Puerto Montt,3.8,78.0,0.5,24.7,2026-05-18
5,La Serena,NaN,61.0,2.6,20.7,2026-01-08
6,Valparaíso,14.6,71.0,0.3,23.5,2026-03-24
7,Puerto Montt,6.0,90.0,0.2,20.5,2026-04-22
8,Antofagasta,26.5,62.0,1.9,28.0,2026-01-15
9,Temuco,10.9,46.0,2.8,12.6,2026-05-03


In [8]:
con = sqlite3.connect(DB_PATH)

df_limpio.to_sql('registros_csv', con, if_exists='replace', index=False)

filas_cargadas = pd.read_sql_query("SELECT COUNT(*) AS total FROM registros_csv", con).iloc[0, 0]
print(f"Filas cargadas en la tabla 'registros_csv': {filas_cargadas}")


Filas cargadas en la tabla 'registros_csv': 190


In [9]:
# Consulta A: ciudad con mayor temperatura promedio
query_a = '''
    SELECT ciudad, AVG(temp_c) AS temp_promedio
    FROM registros_csv
    WHERE temp_c IS NOT NULL
    GROUP BY ciudad
    ORDER BY temp_promedio DESC
    LIMIT 1
'''
resultado_a = pd.read_sql_query(query_a, con)
print("Ciudad con mayor temperatura promedio:")
print(resultado_a)


Ciudad con mayor temperatura promedio:
        ciudad  temp_promedio
0  Antofagasta      20.673333


In [10]:
# Consulta B: promedio de precipitación por ciudad, enero-marzo 2026
query_b = '''
    SELECT ciudad, AVG(precip_mm) AS precip_promedio
    FROM registros_csv
    WHERE fecha BETWEEN '2026-01-01' AND '2026-03-31'
    GROUP BY ciudad
    ORDER BY precip_promedio DESC
'''
resultado_b = pd.read_sql_query(query_b, con)
print("Promedio de precipitación por ciudad (enero-marzo 2026):")
print(resultado_b)


Promedio de precipitación por ciudad (enero-marzo 2026):
         ciudad  precip_promedio
0      Santiago         7.125000
1      Rancagua         3.637500
2     La Serena         3.572727
3        Temuco         3.433333
4   Antofagasta         3.314286
5    Concepción         3.222222
6  Puerto Montt         3.200000
7         Talca         3.138462
8       Iquique         2.360000
9    Valparaíso         2.130000



         ciudad  precip_promedio
0      Santiago         7.125000
1      Rancagua         3.637500
2     La Serena         3.572727
3        Temuco         3.433333
4   Antofagasta         3.314286
5    Concepción         3.222222
6  Puerto Montt         3.200000
7         Talca         3.138462
8       Iquique         2.360000
9    Valparaíso         2.130000


In [11]:
# Consulta C: top 5 fechas con mayor temperatura registrada (cualquier ciudad)
query_c = '''
    SELECT ciudad, fecha, temp_c
    FROM registros_csv
    WHERE temp_c IS NOT NULL
    ORDER BY temp_c DESC
    LIMIT 5
'''
resultado_c = pd.read_sql_query(query_c, con)
print("Top 5 fechas con mayor temperatura registrada:")
print(resultado_c)


Top 5 fechas con mayor temperatura registrada:
        ciudad       fecha  temp_c
0  Antofagasta  2026-04-25    31.7
1      Iquique  2026-03-09    27.6
2  Antofagasta  2026-01-15    26.5
3    La Serena  2026-05-23    24.5
4  Antofagasta  2026-02-18    24.2


### Paso 5 — Resumen final

In [12]:
print("")
print("RESUMEN DEL PIPELINE ETL")
print("")
print(f"Filas originales en el CSV:      {filas_originales}")
print(f"Duplicados eliminados:           {duplicados_eliminados}")
print(f"Valores atípicos eliminados:     {outliers_eliminados}")
print(f"Filas finales cargadas en SQL:   {filas_cargadas}")
print()
print(f"Ciudad más cálida en promedio:   {resultado_a.iloc[0]['ciudad']} "
      f"({resultado_a.iloc[0]['temp_promedio']:.1f}°C)")
print()
print("Promedio de precipitación por ciudad (ene-mar 2026):")
print(resultado_b.to_string(index=False))
print()
print("Top 5 temperaturas más altas registradas:")
print(resultado_c.to_string(index=False))

con.close()



RESUMEN DEL PIPELINE ETL

Filas originales en el CSV:      208
Duplicados eliminados:           13
Valores atípicos eliminados:     5
Filas finales cargadas en SQL:   190

Ciudad más cálida en promedio:   Antofagasta (20.7°C)

Promedio de precipitación por ciudad (ene-mar 2026):
      ciudad  precip_promedio
    Santiago         7.125000
    Rancagua         3.637500
   La Serena         3.572727
      Temuco         3.433333
 Antofagasta         3.314286
  Concepción         3.222222
Puerto Montt         3.200000
       Talca         3.138462
     Iquique         2.360000
  Valparaíso         2.130000

Top 5 temperaturas más altas registradas:
     ciudad      fecha  temp_c
Antofagasta 2026-04-25    31.7
    Iquique 2026-03-09    27.6
Antofagasta 2026-01-15    26.5
  La Serena 2026-05-23    24.5
Antofagasta 2026-02-18    24.2
